## Research Agent 

In [ ]:
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("Research_Agent")
import ast

# Calculator Tool 
@mcp.tool() 
def calculator(exec:str):
    """ user given all aithmetic operations performed by this tool """
    try : 
        return str(ast.literal_eval(exec))
    except Exception as e : 
        return str(e)


In [ ]:
# Weather tool
import requests  
@mcp.tool()
def weather(location :str):
    """ Provided users location's weather fetched by this tool """
    try : 
        response = requests.get(f"https://wttr.in/{location}?format=3").text
        return response
    except Exception as e :
        return str(e)


In [ ]:
# File I/O Operations 
import os
@mcp.tool()
def file_read(filename:str):
    """ file read by this tool """
    with open (file=filename) as f :
        return f" content is : {f.read()}"

@mcp.tool()
def write_file(filename:str,content:str):
    """ file writed by this tool . 
    if file isn't existing then create it and also write the user given content on that
    """
    os.makedirs(os.path.dirname(os.path.abspath(filename)),exist_ok=True)
    with open (file=filename, mode="w") as file : 
        res = file.write(content)
        return f"sucessfully created : {res}" 


In [20]:
# Load document 
from langchain_community.document_loaders import PyPDFLoader 
loader = PyPDFLoader('2005.11401v4.pdf')
pages = loader.load()
# split document  
from langchain_text_splitters import RecursiveCharacterTextSplitter 
spliter = RecursiveCharacterTextSplitter(chunk_overlap=180 , chunk_size=1200)
texts = spliter.split_documents(pages)
chunks = [i.page_content for i in texts]
metadata = [i.metadata for i in texts]

# Use Chromadb 
import chromadb 
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 
embedding_fun = SentenceTransformerEmbeddingFunction("all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path='./Research_Agent')

import warnings 
warnings.filterwarnings('ignore')
try:
    collection = client.get_or_create_collection(name="Agent",embedding_function=embedding_fun)
    if collection.count()==0:
        collection.add(
            ids=[str(i) for i in range(len(chunks))],
            documents=chunks , 
            metadatas=metadata
        )
    print(f"sucessfullt created the collection is : {collection.count()}")
except Exception as e :
    print(str(e))


sucessfullt created the collection is : 71


In [ ]:
# For Hybrid RAG 
from rank_bm25 import BM25Okapi 
token = [i.split() for i in chunks]
token_cor = BM25Okapi(token)

# import LLM 
import os 
from dotenv import load_dotenv 
from langchain_groq import ChatGroq 
load_dotenv()
key=os.getenv("GROQ_API_KEY")
LLM = ChatGroq(model="openai/gpt-oss-120b")


In [ ]:
# Retrival Tool 
import os 
from dotenv import load_dotenv
load_dotenv()
@mcp.tool()
def Hybrid_Rag(query:str):
    """ 
    Before Giving out any answers follow this steps : 
    user's asking questions always follow through these steps
    step 1 :--->  first enter in Hybrid_Rag 
    step 2 :---> search for nearest chunking 
    step 3 :--> got then return the chunks to the user 
    step 4 :---> either goes to the websearch tool [if cant find any related content]
    """
    prompt = f""" 
    refine the user asking and make it context understable : 
    query : {query}
    """
    query_refine = LLM.invoke(prompt).content

    result = collection.query(query_texts=[query_refine],n_results=3)
    distance = result['distances'][0]
    document = result['documents'][0]
    threshold = 1.0
    near_chunks = []

    for dist, doc in zip(distance, document):
        if dist < threshold:
            near_chunks.append(doc)

    score = token_cor.get_scores(query_refine.split())
    def get_scores(score , k=10):
        index  = list(enumerate(score))
        sorted_index = sorted(index , key = lambda x:x[1], reverse=True)
        return [doc for doc , i in sorted_index[:k]]

    get_top_scores = get_scores(score , k=10)
    copy_top_scores = [chunks[i] for i in get_top_scores]

    rrf_token={}

    for rank , doc in enumerate(near_chunks):
        rrf_token[doc]=rrf_token.get(doc,0)+1/(rank+60)

    for rank , doc in enumerate(copy_top_scores):
        rrf_token[doc]=rrf_token.get(doc,0)+1/(rank+60)
    marge = sorted(rrf_token.items() , key=lambda x:x[1] , reverse=True)
    hybrid_top_docs=[doc for doc, _ in marge[:5]]
    if hybrid_top_docs:
        return "\n\n".join(hybrid_top_docs)

    from tavily import TavilyClient

    client = TavilyClient(
        api_key=os.getenv("TAVILY_API_KEY")
    )
    response = client.search(query=query_refine)
    return str(response['results'])


In [ ]:
Hybrid_Rag(
    "What is Retrieval Augmented Generation?"
)


[08/16/26 10:17:13] INFO     HTTP Request: POST https://api.groq.com/openai/v1/chat/completions     _client.py:1025
                             "HTTP/1.1 200 OK"                                                                     

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.96it/s]


'could represent promising future work.\n6 Discussion\nIn this work, we presented hybrid generation models with access to parametric and non-parametric\nmemory. We showed that our RAG models obtain state of the art results on open-domain QA. We\nfound that people prefer RAG’s generation over purely parametric BART, ﬁnding RAG more factual\nand speciﬁc. We conducted an thorough investigation of the learned retrieval component, validating\nits effectiveness, and we illustrated how the retrieval index can be hot-swapped to update the model\nwithout requiring any retraining. In future work, it may be fruitful to investigate if the two components\ncan be jointly pre-trained from scratch, either with a denoising objective similar to BART or some\nanother objective. Our work opens up new research directions on how parametric and non-parametric\nmemories interact and how to most effectively combine them, showing promise in being applied to a\nwide variety of NLP tasks.\n9\n\nthan RAG in only 7

In [ ]:
@mcp.tool()
def Research_tool(query: str):
    docs = Hybrid_Rag(query)

    prompt = f"""
    Answer only the user's question.

    Context:
    {docs}

    Question:
    {query}
    """

    return LLM.invoke(prompt).content

# Runing MCP server
# if __name__ == "__main__":
#     mcp.run()


# Client for Research Agent 

In [ ]:
!uv pip install "mcp==1.13.1"
!uv pip install "langchain-mcp-adapters==0.1.9"


Resolved 25 packages in 50ms                                         
Installed 1 package in 11ms                                 
 + mcp==1.13.1
Resolved 44 packages in 430ms                                        
Prepared 3 packages in 81ms                                                  langchain-core         ------------------------------ 222.81 KiB/450.49 KiB     langchain-core         ------------------------------ 62.81 KiB/450.49 KiB      
Uninstalled 2 packages in 29ms
Installed 3 packages in 6ms86                               
 - langchain-core==1.5.5
 + langchain-core==0.3.86
 + langchain-mcp-adapters==0.1.9
 - packaging==26.3
 + packaging==25.0


In [ ]:
!uv pip install langgraph 


Resolved 35 packages in 127ms                                        
Uninstalled 2 packages in 25ms
Installed 2 packages in 30ms                                
 - langchain-core==0.3.86
 + langchain-core==1.5.5
 - websockets==17.0.1
 + websockets==15.0.1


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient 
import warnings
warnings.filterwarnings('ignore') 

from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver 
memory = MemorySaver()

import os 
from dotenv import load_dotenv 
from langchain_groq import ChatGroq 
load_dotenv()

import asyncio 
async def main ():
    client = MultiServerMCPClient({
        'Research_Agent':{
        'command':'python',
        'args':['Agent_Server.py'],
        'transport':'stdio'
    }
    })

    key=os.getenv("GROQ_API_KEY")
    LLM = ChatGroq(model="openai/gpt-oss-120b") 
    tools = await client.get_tools()
    print([t.name for t in tools])
    if not tool:
        print("tools cant fetch by server now")
    system_prompt = "YOU are a realiable ai assistent for This MCP server so do the work sufficiently one by one using tool"
    research_agent_create=create_react_agent(LLM=LLM , tools=tool , prompt=system_prompt,checkpointer=memory)
    config = {
    "configurable": {
        "thread_id": "research_001"
    }
}

    query = " create a mcp server code in python and file name MCP.txt and print the output  ?"
    print(f'user given query is : {query}') 

    response = await research_agent_create.ainvoke({"messages":[("user",query)]},config=config)
    print(response['messages'][-1].content)


In [ ]:
if __name__=="__main__":
    asyncio.run(main=main())
